In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import requests
import time
import requests
import csv
import pandas as pd
import json
import pandas as pd
import xml.etree.ElementTree as ET
from io import StringIO
from datetime import datetime, timedelta

In [ ]:
myApi = 'YOUR_API_KEY_HERE'

In [4]:
combined = None

In [5]:
origindt = datetime.strptime('20140101', "%Y%m%d")

dt1 = origindt
dt2 = origindt

tm1_str = dt1.strftime("%Y%m%d")
tm2_str = dt2.strftime("%Y%m%d")

savefileName = '3. 관측-통계(일사, 일조) 묶음형 조회서비스'
idontneedhelp = 0

In [7]:
def Get_url(_tm1_str, _tm2_str):
    #return f'https://apihub.kma.go.kr/api/typ01/url/sts_rn.php?tm1={tm1_str}&tm2={tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}' # 14. 강수량 기후통계데이터 조회
    #return f'https://apihub.kma.go.kr/api/typ01/url/sts_ta.php?tm1={tm1_str}&tm2={tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}' # 1. 기온 기후통계데이터 조회
    #return f'https://apihub.kma.go.kr/api/typ01/url/sts_si.php?tm1={tm1_str}&tm2={tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}' # 2. 일사 기후통계데이터 조회
    #return f'https://apihub.kma.go.kr/api/typ01/url/sts_ss.php?tm1={tm1_str}&tm2={tm2_str}&stn_id=0&help={idontneedhelp}&disp=1&authKey={myApi}' #3. 일조 기후통계데이터 조회
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_wind.php?tm1={tm1_str}&tm2={tm2_str}&stn_id=0&help={idontneedhelp}&disp=1&authKey={myApi}'#5. 바람 기후통계데이터 조회
    #return f'https://apihub.kma.go.kr/api/typ01/url/sts_ts.php?tm1={tm1_str}&tm2={tm2_str}&stn_id=0&help={idontneedhelp}&disp=1&authKey={myApi}'
    #return f'https://apihub.kma.go.kr/api/typ01/url/sts_te.php?tm1={tm1_str}&tm2={tm2_str}&stn_id=0&help=0&disp=1&authKey=CH8XAAZ1Qxq_FwAGdcMa8w'  #9. 지중온도 기후통계데이터 조회
    #return f'https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph-sun_sfc_sts_pkg?authKey={myApi}&stn=0&tm1={tm1_str}&tm2={tm2_str}&mode=si&help=0&disp=0'#3. 관측-통계(일사, 일조) 묶음형 조회서비스

In [ ]:
# @title 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:  # 첫 번째 줄(헤더)을 제외한 모든 줄
        line_data = line.strip()

        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()

        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])

        if len(_column_names) != cols_in_data:
            print(f"경고: 열 이름({len(_column_names)}개)과 데이터 열({cols_in_data}개)의 개수 불일치.")

            if len(_column_names) > cols_in_data:
                # 열 이름이 더 많은 경우, 데이터에 맞게 열 이름 조정
                _column_names = _column_names[:cols_in_data]
                print(f"열 이름을 데이터에 맞게 조정합니다: {_column_names}")
            else:
                # 데이터 열이 더 많은 경우, 추가 열 이름 생성
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)
                print(f"추가 열 이름을 생성합니다: {extra_cols}")

    # DataFrame 생성
    _df = pd.DataFrame(data, columns=_column_names)
    return _df

In [ ]:
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)

        if response.status_code == 200:
            break
        else:
            print(f'code : {response.status_code}')
        # time.sleep(1)


    # 각 줄로 분리
    text_data = response.text
    lines = text_data.strip().split('\n')

    # del lines[0]
    # del lines[0]
    # del lines[1]
    # del lines[1]
    # del lines[-1]

    # 첫 번째 줄에서 열 이름 추출
    header_line = lines[0].strip()

    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()

    # 헤더 라인의 '#' 기호 처리
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()

    df = Set_ConvertData(lines, column_names)

    combined  = pd.concat([combined, df])

    dt1 = dt2 + timedelta(days=1)
    print(dt1.strftime("%Y%m%d"))

20140114
20140127


KeyboardInterrupt: 

In [ ]:
combined

In [ ]:
combined.to_csv(savefileName + '.csv')

In [ ]:
import numpy as np
import pandas as pd
import requests
import time
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_일사_일조_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수
def Get_url(_tm1_str, _tm2_str):
    # 아래 항목 중 하나를 사용하려면 주석을 해제하세요:

    # ① 강수량 통계
    # return f'https://apihub.kma.go.kr/api/typ01/url/sts_rn.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

    # ② 기온 통계
    # return f'https://apihub.kma.go.kr/api/typ01/url/sts_ta.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

    # ③ 일사 통계
    # return f'https://apihub.kma.go.kr/api/typ01/url/sts_si.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

    # ④ 일조 통계
    # return f'https://apihub.kma.go.kr/api/typ01/url/sts_ss.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help={idontneedhelp}&disp=1&authKey={myApi}'

    # ⑤ 바람 통계 (현재 선택됨)
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_wind.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help={idontneedhelp}&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드 (지점명 매핑)
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 데이터 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 지점 ID 컬럼명 통일
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    # 지점별로 저장
    for stn_id, df_stn in combined.groupby('STN_ID'):
        name = meta.loc[meta['지점'] == int(stn_id), '지점명']
        stn_name = name.iloc[0] if not name.empty else str(stn_id)
        filename = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
        df_stn.to_csv(filename, index=False, encoding='utf-8-sig')

    print("✅ 지점별 파일 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


✔ 20140101 ~ 20140113 수집 완료
✔ 20140114 ~ 20140126 수집 완료
✔ 20140127 ~ 20140208 수집 완료
✔ 20140209 ~ 20140221 수집 완료
✔ 20140222 ~ 20140306 수집 완료
✔ 20140307 ~ 20140319 수집 완료
✔ 20140320 ~ 20140401 수집 완료
✔ 20140402 ~ 20140414 수집 완료
✔ 20140415 ~ 20140427 수집 완료
✔ 20140428 ~ 20140510 수집 완료
✔ 20140511 ~ 20140523 수집 완료
✔ 20140524 ~ 20140605 수집 완료
✔ 20140606 ~ 20140618 수집 완료
✔ 20140619 ~ 20140701 수집 완료
✔ 20140702 ~ 20140714 수집 완료
✔ 20140715 ~ 20140727 수집 완료
✔ 20140728 ~ 20140809 수집 완료
✔ 20140810 ~ 20140822 수집 완료
✔ 20140823 ~ 20140904 수집 완료
✔ 20140905 ~ 20140917 수집 완료
✔ 20140918 ~ 20140930 수집 완료
✔ 20141001 ~ 20141013 수집 완료
✔ 20141014 ~ 20141026 수집 완료
✔ 20141027 ~ 20141108 수집 완료
✔ 20141109 ~ 20141121 수집 완료
✔ 20141122 ~ 20141204 수집 완료
✔ 20141205 ~ 20141217 수집 완료
✔ 20141218 ~ 20141230 수집 완료
✔ 20141231 ~ 20150112 수집 완료
✔ 20150113 ~ 20150125 수집 완료
✔ 20150126 ~ 20150207 수집 완료
✔ 20150208 ~ 20150220 수집 완료
✔ 20150221 ~ 20150305 수집 완료
✔ 20150306 ~ 20150318 수집 완료
✔ 20150319 ~ 20150331 수집 완료
✔ 20150401 ~ 2015041

In [11]:
import zipfile
import os

# 압축할 파일들이 있는 디렉토리 (현재 디렉토리일 경우 '.')
target_dir = '.'
zip_filename = '기상청_관측소별_데이터.zip'

# ZIP 파일 생성
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in os.listdir(target_dir):
        if file.endswith('.csv') and savefileName in file:
            zipf.write(os.path.join(target_dir, file), arcname=file)

print(f"✅ 모든 CSV 파일이 {zip_filename}으로 압축되었습니다.")


✅ 모든 CSV 파일이 기상청_관측소별_데이터.zip으로 압축되었습니다.


In [14]:
import zipfile
import os
import re

zip_filename = '기상청_관측소별_데이터.zip'

# "숫자_한글포함_..." 형태의 파일만 필터링 (한글/공백/쉼표 허용)
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in os.listdir('.'):
        if pattern.match(file):
            zipf.write(file, arcname=file)

print(f"✅ 숫자_한글형식(.csv)만 포함된 {zip_filename} 생성 완료")


✅ 숫자_한글형식(.csv)만 포함된 기상청_관측소별_데이터.zip 생성 완료


In [15]:
import os
import re
import pandas as pd

# 정규표현식: 숫자_한글_ 으로 시작하는 CSV만 포함
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

# 포함된 파일 목록 추출
csv_files = [f for f in os.listdir('./') if pattern.match(f)]

# 파일 경로 생성
csv_paths = [os.path.join('./', f) for f in csv_files]

# CSV 파일 병합
merged_df = pd.concat([pd.read_csv(path) for path in csv_paths], ignore_index=True)

# 저장
merged_path = './merged_filtered_stations.csv'
merged_df.to_csv(merged_path, index=False, encoding='utf-8-sig')

merged_path


'./merged_filtered_stations.csv'

강수량통계

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import requests
import time
import requests
import csv
import pandas as pd
import json
import pandas as pd
import xml.etree.ElementTree as ET
from io import StringIO
from datetime import datetime, timedelta

In [3]:
#############################이전 프로젝트에 썼던 코드################################
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm  # 폰트 관리

!apt-get update -qq         # apt-get 패키지 설치 명령어, -qq : 에러외 메세지 숨기기
!apt-get install fonts-nanum* -qq #나눔글꼴 설치

fe = fm.FontEntry(fname=r'/usr/share/fonts/truetype/nanum/NanumGothic.ttf', name='NanumGothic') #파일 저장되어있는 경로와 이름 설정
fm.fontManager.ttflist.insert(0, fe)  # Matplotlib에 폰트 추가
plt.rcParams.update({'font.size': 10, 'font.family': 'NanumGothic'}) #폰트설정

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-nanum.
(Reading database ... 126102 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Selecting previously unselected package fonts-nanum-coding.
Preparing to unpack .../fonts-nanum-coding_2.5-3_all.deb ...
Unpacking fonts-nanum-coding (2.5-3) ...
Selecting previously unselected package fonts-nanum-eco.
Preparing to unpack .../fonts-nanum-eco_1.000-7_all.deb ...
Unpacking fonts-nanum-eco (1.000-7) ...
Selecting previously unselected package fonts-nanum-extra.
Preparing to unpack .../fonts-nanum-extra_20200506-1_all.deb ...
Unpacking fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Setting up fo

In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import zipfile
import os
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_강수량_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수 - 강수량 통계
def Get_url(_tm1_str, _tm2_str):
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_rn.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 컬럼 통일
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    # 지점별로 나눠 zip에 저장
    with zipfile.ZipFile(f"{savefileName}.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for stn_id, df_stn in combined.groupby('STN_ID'):
            name = meta.loc[meta['지점'] == int(stn_id), '지점명']
            stn_name = name.iloc[0] if not name.empty else str(stn_id)
            file_name = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
            df_stn.to_csv(file_name, index=False, encoding='utf-8-sig')
            zf.write(file_name)
            os.remove(file_name)

    print("✅ ZIP 파일로만 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


✔ 20140101 ~ 20140113 수집 완료
✔ 20140114 ~ 20140126 수집 완료
✔ 20140127 ~ 20140208 수집 완료
✔ 20140209 ~ 20140221 수집 완료
✔ 20140222 ~ 20140306 수집 완료
✔ 20140307 ~ 20140319 수집 완료
✔ 20140320 ~ 20140401 수집 완료
✔ 20140402 ~ 20140414 수집 완료
✔ 20140415 ~ 20140427 수집 완료
✔ 20140428 ~ 20140510 수집 완료
✔ 20140511 ~ 20140523 수집 완료
✔ 20140524 ~ 20140605 수집 완료
✔ 20140606 ~ 20140618 수집 완료
✔ 20140619 ~ 20140701 수집 완료
✔ 20140702 ~ 20140714 수집 완료
✔ 20140715 ~ 20140727 수집 완료
✔ 20140728 ~ 20140809 수집 완료
✔ 20140810 ~ 20140822 수집 완료
✔ 20140823 ~ 20140904 수집 완료
✔ 20140905 ~ 20140917 수집 완료
✔ 20140918 ~ 20140930 수집 완료
✔ 20141001 ~ 20141013 수집 완료
✔ 20141014 ~ 20141026 수집 완료
✔ 20141027 ~ 20141108 수집 완료
✔ 20141109 ~ 20141121 수집 완료
✔ 20141122 ~ 20141204 수집 완료
✔ 20141205 ~ 20141217 수집 완료
✔ 20141218 ~ 20141230 수집 완료
✔ 20141231 ~ 20150112 수집 완료
✔ 20150113 ~ 20150125 수집 완료
✔ 20150126 ~ 20150207 수집 완료
✔ 20150208 ~ 20150220 수집 완료
✔ 20150221 ~ 20150305 수집 완료
✔ 20150306 ~ 20150318 수집 완료
✔ 20150319 ~ 20150331 수집 완료
✔ 20150401 ~ 2015041

In [8]:
import zipfile
import re
import pandas as pd
import io

# ZIP 파일 경로
zip_path = './관측소별_강수량_통계.zip'

# 정규표현식: 숫자_한글_ 형식 CSV
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

# 병합용 리스트
dfs = []

with zipfile.ZipFile(zip_path, 'r') as zf:
    for file_name in zf.namelist():
        if pattern.match(file_name):
            with zf.open(file_name) as f:
                df = pd.read_csv(f)
                dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)

# 저장
output_path = './merged_from_zip.csv'
merged_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'✅ 저장 완료: {output_path}')


✅ 저장 완료: ./merged_from_zip.csv


In [9]:
import zipfile
import re

# 원본 ZIP 경로
input_zip = './관측소별_강수량_통계.zip'
# 출력 ZIP 경로
output_zip = './관측소별_강수량_한글only.zip'

# 한글 이름 포함 형식: 000_한글_*.csv
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

with zipfile.ZipFile(input_zip, 'r') as zin:
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        for file_name in zin.namelist():
            if pattern.match(file_name):
                # 원본 zip에서 읽어서 그대로 복사
                data = zin.read(file_name)
                zout.writestr(file_name, data)

print(f'✅ 한글 이름만 포함된 zip 생성 완료: {output_zip}')


✅ 한글 이름만 포함된 zip 생성 완료: ./관측소별_강수량_한글only.zip


In [10]:
import zipfile
import pandas as pd
import re
import io

# 입력 ZIP 경로 (한글only)
input_zip = './관측소별_강수량_한글only.zip'
output_csv = './merged_from_korean_named_zip.csv'

# 숫자_한글_ 형식 필터 정규표현식 (이미 한글only지만 다시 확인)
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

dfs = []

with zipfile.ZipFile(input_zip, 'r') as zf:
    for file_name in zf.namelist():
        if pattern.match(file_name):
            with zf.open(file_name) as f:
                df = pd.read_csv(f)
                dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'✅ 병합된 CSV 저장 완료: {output_csv}')


✅ 병합된 CSV 저장 완료: ./merged_from_korean_named_zip.csv


In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import zipfile
import os
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_강수량_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수 - 강수량 통계
def Get_url(_tm1_str, _tm2_str):
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_rn.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 컬럼 통일
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    # 지점별로 나눠 zip에 저장
    with zipfile.ZipFile(f"{savefileName}.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for stn_id, df_stn in combined.groupby('STN_ID'):
            name = meta.loc[meta['지점'] == int(stn_id), '지점명']
            stn_name = name.iloc[0] if not name.empty else str(stn_id)
            file_name = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
            df_stn.to_csv(file_name, index=False, encoding='utf-8-sig')
            zf.write(file_name)
            os.remove(file_name)

    print("✅ ZIP 파일로만 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import zipfile
import os
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_기온_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수 - 기온 통계
def Get_url(_tm1_str, _tm2_str):
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_ta.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 컬럼 통일 및 지점별로 ZIP 저장
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    with zipfile.ZipFile(f"{savefileName}.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for stn_id, df_stn in combined.groupby('STN_ID'):
            name = meta.loc[meta['지점'] == int(stn_id), '지점명']
            stn_name = name.iloc[0] if not name.empty else str(stn_id)
            file_name = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
            df_stn.to_csv(file_name, index=False, encoding='utf-8-sig')
            zf.write(file_name)
            os.remove(file_name)

    print("✅ 기온 통계 ZIP 파일로만 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


✔ 20140101 ~ 20140113 수집 완료
✔ 20140114 ~ 20140126 수집 완료
✔ 20140127 ~ 20140208 수집 완료
✔ 20140209 ~ 20140221 수집 완료
✔ 20140222 ~ 20140306 수집 완료
✔ 20140307 ~ 20140319 수집 완료
✔ 20140320 ~ 20140401 수집 완료
✔ 20140402 ~ 20140414 수집 완료
✔ 20140415 ~ 20140427 수집 완료
✔ 20140428 ~ 20140510 수집 완료
✔ 20140511 ~ 20140523 수집 완료
✔ 20140524 ~ 20140605 수집 완료
✔ 20140606 ~ 20140618 수집 완료
✔ 20140619 ~ 20140701 수집 완료
✔ 20140702 ~ 20140714 수집 완료
✔ 20140715 ~ 20140727 수집 완료
✔ 20140728 ~ 20140809 수집 완료
✔ 20140810 ~ 20140822 수집 완료
✔ 20140823 ~ 20140904 수집 완료
✔ 20140905 ~ 20140917 수집 완료
✔ 20140918 ~ 20140930 수집 완료
✔ 20141001 ~ 20141013 수집 완료
✔ 20141014 ~ 20141026 수집 완료
✔ 20141027 ~ 20141108 수집 완료
✔ 20141109 ~ 20141121 수집 완료
✔ 20141122 ~ 20141204 수집 완료
✔ 20141205 ~ 20141217 수집 완료
✔ 20141218 ~ 20141230 수집 완료
✔ 20141231 ~ 20150112 수집 완료
✔ 20150113 ~ 20150125 수집 완료
✔ 20150126 ~ 20150207 수집 완료
✔ 20150208 ~ 20150220 수집 완료
✔ 20150221 ~ 20150305 수집 완료
✔ 20150306 ~ 20150318 수집 완료
✔ 20150319 ~ 20150331 수집 완료
✔ 20150401 ~ 2015041

In [13]:
import zipfile
import re

# 원본 ZIP 경로
input_zip = './관측소별_기온_통계.zip'
# 출력 ZIP 경로
output_zip = './관측소별_기온_한글only.zip'

# 한글 이름 포함 형식: 000_한글_*.csv
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

with zipfile.ZipFile(input_zip, 'r') as zin:
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        for file_name in zin.namelist():
            if pattern.match(file_name):
                # 원본 zip에서 읽어서 그대로 복사
                data = zin.read(file_name)
                zout.writestr(file_name, data)

print(f'✅ 한글 이름만 포함된 zip 생성 완료: {output_zip}')


✅ 한글 이름만 포함된 zip 생성 완료: ./관측소별_기온_한글only.zip


In [14]:
import zipfile
import pandas as pd
import re
import io

# 입력 ZIP 경로 (한글only)
input_zip = './관측소별_기온_한글only.zip'
output_csv = './기온데이터_merged_from_korean_named_zip.csv'

# 숫자_한글_ 형식 필터 정규표현식 (이미 한글only지만 다시 확인)
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

dfs = []

with zipfile.ZipFile(input_zip, 'r') as zf:
    for file_name in zf.namelist():
        if pattern.match(file_name):
            with zf.open(file_name) as f:
                df = pd.read_csv(f)
                dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'✅ 병합된 CSV 저장 완료: {output_csv}')


✅ 병합된 CSV 저장 완료: ./기온데이터_merged_from_korean_named_zip.csv


In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import zipfile
import os
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_습도_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수 - 습도 통계
def Get_url(_tm1_str, _tm2_str):
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_rhm.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 컬럼 통일 및 지점별 ZIP 저장
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    with zipfile.ZipFile(f"{savefileName}.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for stn_id, df_stn in combined.groupby('STN_ID'):
            name = meta.loc[meta['지점'] == int(stn_id), '지점명']
            stn_name = name.iloc[0] if not name.empty else str(stn_id)
            file_name = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
            df_stn.to_csv(file_name, index=False, encoding='utf-8-sig')
            zf.write(file_name)
            os.remove(file_name)

    print("✅ 습도 통계 ZIP 파일로만 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


✔ 20140101 ~ 20140113 수집 완료
✔ 20140114 ~ 20140126 수집 완료
✔ 20140127 ~ 20140208 수집 완료
✔ 20140209 ~ 20140221 수집 완료
✔ 20140222 ~ 20140306 수집 완료
✔ 20140307 ~ 20140319 수집 완료
✔ 20140320 ~ 20140401 수집 완료
✔ 20140402 ~ 20140414 수집 완료
✔ 20140415 ~ 20140427 수집 완료
✔ 20140428 ~ 20140510 수집 완료
✔ 20140511 ~ 20140523 수집 완료
✔ 20140524 ~ 20140605 수집 완료
✔ 20140606 ~ 20140618 수집 완료
✔ 20140619 ~ 20140701 수집 완료
✔ 20140702 ~ 20140714 수집 완료
✔ 20140715 ~ 20140727 수집 완료
✔ 20140728 ~ 20140809 수집 완료
✔ 20140810 ~ 20140822 수집 완료
✔ 20140823 ~ 20140904 수집 완료
✔ 20140905 ~ 20140917 수집 완료
✔ 20140918 ~ 20140930 수집 완료
✔ 20141001 ~ 20141013 수집 완료
✔ 20141014 ~ 20141026 수집 완료
✔ 20141027 ~ 20141108 수집 완료
✔ 20141109 ~ 20141121 수집 완료
✔ 20141122 ~ 20141204 수집 완료
✔ 20141205 ~ 20141217 수집 완료
✔ 20141218 ~ 20141230 수집 완료
✔ 20141231 ~ 20150112 수집 완료
✔ 20150113 ~ 20150125 수집 완료
✔ 20150126 ~ 20150207 수집 완료
✔ 20150208 ~ 20150220 수집 완료
✔ 20150221 ~ 20150305 수집 완료
✔ 20150306 ~ 20150318 수집 완료
✔ 20150319 ~ 20150331 수집 완료
✔ 20150401 ~ 2015041

In [16]:
import zipfile
import re

# 원본 ZIP 경로
input_zip = './관측소별_습도_통계.zip'
# 출력 ZIP 경로
output_zip = './관측소별_습도_한글only.zip'

# 한글 이름 포함 형식: 000_한글_*.csv
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

with zipfile.ZipFile(input_zip, 'r') as zin:
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        for file_name in zin.namelist():
            if pattern.match(file_name):
                # 원본 zip에서 읽어서 그대로 복사
                data = zin.read(file_name)
                zout.writestr(file_name, data)

print(f'✅ 한글 이름만 포함된 zip 생성 완료: {output_zip}')


✅ 한글 이름만 포함된 zip 생성 완료: ./관측소별_습도_한글only.zip


In [17]:
import zipfile
import pandas as pd
import re
import io

# 입력 ZIP 경로 (한글only)
input_zip = './관측소별_습도_한글only.zip'
output_csv = './습도데이터_merged_from_korean_named_zip.csv'

# 숫자_한글_ 형식 필터 정규표현식 (이미 한글only지만 다시 확인)
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

dfs = []

with zipfile.ZipFile(input_zip, 'r') as zf:
    for file_name in zf.namelist():
        if pattern.match(file_name):
            with zf.open(file_name) as f:
                df = pd.read_csv(f)
                dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'✅ 병합된 CSV 저장 완료: {output_csv}')


✅ 병합된 CSV 저장 완료: ./습도데이터_merged_from_korean_named_zip.csv


일사

In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import zipfile
import os
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_일사_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수 - 일사 통계
def Get_url(_tm1_str, _tm2_str):
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_si.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 컬럼 통일 및 ZIP 저장
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    with zipfile.ZipFile(f"{savefileName}.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for stn_id, df_stn in combined.groupby('STN_ID'):
            name = meta.loc[meta['지점'] == int(stn_id), '지점명']
            stn_name = name.iloc[0] if not name.empty else str(stn_id)
            file_name = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
            df_stn.to_csv(file_name, index=False, encoding='utf-8-sig')
            zf.write(file_name)
            os.remove(file_name)

    print("✅ 일사 통계 ZIP 파일로만 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


✔ 20140101 ~ 20140113 수집 완료
✔ 20140114 ~ 20140126 수집 완료
✔ 20140127 ~ 20140208 수집 완료
✔ 20140209 ~ 20140221 수집 완료
✔ 20140222 ~ 20140306 수집 완료
✔ 20140307 ~ 20140319 수집 완료
✔ 20140320 ~ 20140401 수집 완료
✔ 20140402 ~ 20140414 수집 완료
✔ 20140415 ~ 20140427 수집 완료
✔ 20140428 ~ 20140510 수집 완료
✔ 20140511 ~ 20140523 수집 완료
✔ 20140524 ~ 20140605 수집 완료
✔ 20140606 ~ 20140618 수집 완료
✔ 20140619 ~ 20140701 수집 완료
✔ 20140702 ~ 20140714 수집 완료
✔ 20140715 ~ 20140727 수집 완료
✔ 20140728 ~ 20140809 수집 완료
✔ 20140810 ~ 20140822 수집 완료
✔ 20140823 ~ 20140904 수집 완료
✔ 20140905 ~ 20140917 수집 완료
✔ 20140918 ~ 20140930 수집 완료
✔ 20141001 ~ 20141013 수집 완료
✔ 20141014 ~ 20141026 수집 완료
✔ 20141027 ~ 20141108 수집 완료
✔ 20141109 ~ 20141121 수집 완료
✔ 20141122 ~ 20141204 수집 완료
✔ 20141205 ~ 20141217 수집 완료
✔ 20141218 ~ 20141230 수집 완료
✔ 20141231 ~ 20150112 수집 완료
✔ 20150113 ~ 20150125 수집 완료
✔ 20150126 ~ 20150207 수집 완료
✔ 20150208 ~ 20150220 수집 완료
✔ 20150221 ~ 20150305 수집 완료
✔ 20150306 ~ 20150318 수집 완료
✔ 20150319 ~ 20150331 수집 완료
✔ 20150401 ~ 2015041

In [19]:
import zipfile
import re

# 원본 ZIP 경로
input_zip = './관측소별_일사_통계.zip'
# 출력 ZIP 경로
output_zip = './관측소별_일사_한글only.zip'

# 한글 이름 포함 형식: 000_한글_*.csv
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

with zipfile.ZipFile(input_zip, 'r') as zin:
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        for file_name in zin.namelist():
            if pattern.match(file_name):
                # 원본 zip에서 읽어서 그대로 복사
                data = zin.read(file_name)
                zout.writestr(file_name, data)

print(f'✅ 한글 이름만 포함된 zip 생성 완료: {output_zip}')


✅ 한글 이름만 포함된 zip 생성 완료: ./관측소별_일사_한글only.zip


In [20]:
import zipfile
import pandas as pd
import re
import io

# 입력 ZIP 경로 (한글only)
input_zip = './관측소별_일사_한글only.zip'
output_csv = './일사데이터_merged_from_korean_named_zip.csv'

# 숫자_한글_ 형식 필터 정규표현식 (이미 한글only지만 다시 확인)
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

dfs = []

with zipfile.ZipFile(input_zip, 'r') as zf:
    for file_name in zf.namelist():
        if pattern.match(file_name):
            with zf.open(file_name) as f:
                df = pd.read_csv(f)
                dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'✅ 병합된 CSV 저장 완료: {output_csv}')


✅ 병합된 CSV 저장 완료: ./일사데이터_merged_from_korean_named_zip.csv


이슬점

In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import zipfile
import os
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_이슬점_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수 - 이슬점 통계
def Get_url(_tm1_str, _tm2_str):
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_td.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 컬럼 통일 및 ZIP 저장
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    with zipfile.ZipFile(f"{savefileName}.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for stn_id, df_stn in combined.groupby('STN_ID'):
            name = meta.loc[meta['지점'] == int(stn_id), '지점명']
            stn_name = name.iloc[0] if not name.empty else str(stn_id)
            file_name = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
            df_stn.to_csv(file_name, index=False, encoding='utf-8-sig')
            zf.write(file_name)
            os.remove(file_name)

    print("✅ 이슬점 통계 ZIP 파일로만 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


오류 코드: 502
✔ 20140101 ~ 20140113 수집 완료
✔ 20140114 ~ 20140126 수집 완료
✔ 20140127 ~ 20140208 수집 완료
✔ 20140209 ~ 20140221 수집 완료
✔ 20140222 ~ 20140306 수집 완료
✔ 20140307 ~ 20140319 수집 완료
✔ 20140320 ~ 20140401 수집 완료
✔ 20140402 ~ 20140414 수집 완료
✔ 20140415 ~ 20140427 수집 완료
✔ 20140428 ~ 20140510 수집 완료
✔ 20140511 ~ 20140523 수집 완료
✔ 20140524 ~ 20140605 수집 완료
✔ 20140606 ~ 20140618 수집 완료
✔ 20140619 ~ 20140701 수집 완료
✔ 20140702 ~ 20140714 수집 완료
✔ 20140715 ~ 20140727 수집 완료
✔ 20140728 ~ 20140809 수집 완료
✔ 20140810 ~ 20140822 수집 완료
✔ 20140823 ~ 20140904 수집 완료
✔ 20140905 ~ 20140917 수집 완료
✔ 20140918 ~ 20140930 수집 완료
✔ 20141001 ~ 20141013 수집 완료
✔ 20141014 ~ 20141026 수집 완료
✔ 20141027 ~ 20141108 수집 완료
✔ 20141109 ~ 20141121 수집 완료
✔ 20141122 ~ 20141204 수집 완료
✔ 20141205 ~ 20141217 수집 완료
✔ 20141218 ~ 20141230 수집 완료
✔ 20141231 ~ 20150112 수집 완료
✔ 20150113 ~ 20150125 수집 완료
✔ 20150126 ~ 20150207 수집 완료
✔ 20150208 ~ 20150220 수집 완료
✔ 20150221 ~ 20150305 수집 완료
✔ 20150306 ~ 20150318 수집 완료
✔ 20150319 ~ 20150331 수집 완료
✔ 2015040

In [22]:
import zipfile
import re

# 원본 ZIP 경로
input_zip = './관측소별_이슬점_통계.zip'
# 출력 ZIP 경로
output_zip = './관측소별_이슬점_한글only.zip'

# 한글 이름 포함 형식: 000_한글_*.csv
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

with zipfile.ZipFile(input_zip, 'r') as zin:
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        for file_name in zin.namelist():
            if pattern.match(file_name):
                # 원본 zip에서 읽어서 그대로 복사
                data = zin.read(file_name)
                zout.writestr(file_name, data)

print(f'✅ 한글 이름만 포함된 zip 생성 완료: {output_zip}')


✅ 한글 이름만 포함된 zip 생성 완료: ./관측소별_이슬점_한글only.zip


In [23]:
import zipfile
import pandas as pd
import re
import io

# 입력 ZIP 경로 (한글only)
input_zip = './관측소별_이슬점_한글only.zip'
output_csv = './이슬점데이터_merged_from_korean_named_zip.csv'

# 숫자_한글_ 형식 필터 정규표현식 (이미 한글only지만 다시 확인)
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

dfs = []

with zipfile.ZipFile(input_zip, 'r') as zf:
    for file_name in zf.namelist():
        if pattern.match(file_name):
            with zf.open(file_name) as f:
                df = pd.read_csv(f)
                dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'✅ 병합된 CSV 저장 완료: {output_csv}')


✅ 병합된 CSV 저장 완료: ./이슬점데이터_merged_from_korean_named_zip.csv


In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import zipfile
import os
from datetime import datetime, timedelta

# 인증키 설정
myApi = 'YOUR_API_KEY_HERE'

# 날짜 초기화
origindt = datetime.strptime('20140101', "%Y%m%d")
dt1 = origindt
dt2 = origindt

savefileName = '관측소별_증발량_통계'
idontneedhelp = 0
combined = None

# URL 생성 함수 - 증발량 통계
def Get_url(_tm1_str, _tm2_str):
    return f'https://apihub.kma.go.kr/api/typ01/url/sts_ev.php?tm1={_tm1_str}&tm2={_tm2_str}&stn_id=0&help=0&disp=1&authKey={myApi}'

# 데이터 정제 함수
def Set_ConvertData(_lines, _column_names):
    data = []
    for line in _lines[1:]:
        line_data = line.strip()
        if '=' in line_data:
            line_data = line_data.split('=')[0].strip()
        values = line_data.split()
        if values:
            data.append(values)

    if data:
        cols_in_data = len(data[0])
        if len(_column_names) != cols_in_data:
            if len(_column_names) > cols_in_data:
                _column_names = _column_names[:cols_in_data]
            else:
                extra_cols = [f'Column_{i+1}' for i in range(len(_column_names), cols_in_data)]
                _column_names.extend(extra_cols)

    return pd.DataFrame(data, columns=_column_names)

# 지점 메타파일 로드
meta_path = './META_관측지점정보_20250429060006.csv'
meta = pd.read_csv(meta_path, encoding='euc-kr')
meta = meta[['지점', '지점명']].dropna()

# 수집 루프
while dt1 <= datetime.strptime('20250101', "%Y%m%d"):
    dt2 = dt1 + timedelta(days=12)
    tm1_str = dt1.strftime("%Y%m%d")
    tm2_str = dt2.strftime("%Y%m%d")

    url = Get_url(tm1_str, tm2_str)
    response = None

    while True:
        response = requests.get(url)
        if response.status_code == 200:
            break
        else:
            print(f'오류 코드: {response.status_code}')
            time.sleep(1)

    lines = response.text.strip().split('\n')
    if len(lines) < 2:
        dt1 = dt2 + timedelta(days=1)
        continue

    header_line = lines[0].strip()
    if '=' in header_line:
        header_line = header_line.split('=')[0].strip()
    if header_line.startswith('#'):
        header_line = header_line[1:].strip()

    column_names = header_line.split()
    df = Set_ConvertData(lines, column_names)

    if combined is None:
        combined = df
    else:
        combined = pd.concat([combined, df], ignore_index=True)

    print(f"✔ {tm1_str} ~ {tm2_str} 수집 완료")
    dt1 = dt2 + timedelta(days=1)

# 컬럼 통일 및 ZIP 저장
if combined is not None:
    for possible_col in ['STNID', '지점']:
        if possible_col in combined.columns:
            combined.rename(columns={possible_col: 'STN_ID'}, inplace=True)

    with zipfile.ZipFile(f"{savefileName}.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for stn_id, df_stn in combined.groupby('STN_ID'):
            name = meta.loc[meta['지점'] == int(stn_id), '지점명']
            stn_name = name.iloc[0] if not name.empty else str(stn_id)
            file_name = f"{int(stn_id):03d}_{stn_name}_{savefileName}.csv"
            df_stn.to_csv(file_name, index=False, encoding='utf-8-sig')
            zf.write(file_name)
            os.remove(file_name)

    print("✅ 증발량 통계 ZIP 파일로만 저장 완료!")
else:
    print("⚠️ 수집된 데이터가 없습니다.")


✔ 20140101 ~ 20140113 수집 완료
✔ 20140114 ~ 20140126 수집 완료
✔ 20140127 ~ 20140208 수집 완료
✔ 20140209 ~ 20140221 수집 완료
✔ 20140222 ~ 20140306 수집 완료
✔ 20140307 ~ 20140319 수집 완료
✔ 20140320 ~ 20140401 수집 완료
✔ 20140402 ~ 20140414 수집 완료
✔ 20140415 ~ 20140427 수집 완료
✔ 20140428 ~ 20140510 수집 완료
✔ 20140511 ~ 20140523 수집 완료
✔ 20140524 ~ 20140605 수집 완료
✔ 20140606 ~ 20140618 수집 완료
✔ 20140619 ~ 20140701 수집 완료
✔ 20140702 ~ 20140714 수집 완료
✔ 20140715 ~ 20140727 수집 완료
✔ 20140728 ~ 20140809 수집 완료
✔ 20140810 ~ 20140822 수집 완료
✔ 20140823 ~ 20140904 수집 완료
✔ 20140905 ~ 20140917 수집 완료
✔ 20140918 ~ 20140930 수집 완료
✔ 20141001 ~ 20141013 수집 완료
✔ 20141014 ~ 20141026 수집 완료
✔ 20141027 ~ 20141108 수집 완료
✔ 20141109 ~ 20141121 수집 완료
✔ 20141122 ~ 20141204 수집 완료
✔ 20141205 ~ 20141217 수집 완료
✔ 20141218 ~ 20141230 수집 완료
✔ 20141231 ~ 20150112 수집 완료
✔ 20150113 ~ 20150125 수집 완료
✔ 20150126 ~ 20150207 수집 완료
✔ 20150208 ~ 20150220 수집 완료
✔ 20150221 ~ 20150305 수집 완료
✔ 20150306 ~ 20150318 수집 완료
✔ 20150319 ~ 20150331 수집 완료
✔ 20150401 ~ 2015041

In [25]:
import zipfile
import re

# 원본 ZIP 경로
input_zip = './관측소별_증발량_통계.zip'
# 출력 ZIP 경로
output_zip = './관측소별_증발량_한글only.zip'

# 한글 이름 포함 형식: 000_한글_*.csv
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

with zipfile.ZipFile(input_zip, 'r') as zin:
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        for file_name in zin.namelist():
            if pattern.match(file_name):
                # 원본 zip에서 읽어서 그대로 복사
                data = zin.read(file_name)
                zout.writestr(file_name, data)

print(f'✅ 한글 이름만 포함된 zip 생성 완료: {output_zip}')


✅ 한글 이름만 포함된 zip 생성 완료: ./관측소별_증발량_한글only.zip


In [26]:
import zipfile
import pandas as pd
import re
import io

# 입력 ZIP 경로 (한글only)
input_zip = './관측소별_증발량_한글only.zip'
output_csv = './증발량데이터_merged_from_korean_named_zip.csv'

# 숫자_한글_ 형식 필터 정규표현식 (이미 한글only지만 다시 확인)
pattern = re.compile(r'^\d{3}_[가-힣 ,]+_.*\.csv$')

dfs = []

with zipfile.ZipFile(input_zip, 'r') as zf:
    for file_name in zf.namelist():
        if pattern.match(file_name):
            with zf.open(file_name) as f:
                df = pd.read_csv(f)
                dfs.append(df)

# 병합
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'✅ 병합된 CSV 저장 완료: {output_csv}')


✅ 병합된 CSV 저장 완료: ./증발량데이터_merged_from_korean_named_zip.csv
